# Carregando Modelo

In [1]:
import torch
from torchvision import models

In [2]:
device = "cpu"
model_path = "Models/Saved/Expert/Corn/corn.pth"

In [3]:
NET = models.convnext_tiny(weights=None)
in_features = NET.classifier[2].in_features

LOADED_CHECKPOINT = torch.load(
    model_path,
    map_location=device,
    )

In [4]:
if LOADED_CHECKPOINT["model_state_dict"] != None:
    NET.classifier[2] = torch.nn.Linear(in_features, LOADED_CHECKPOINT["num_classes"])

    NET.load_state_dict(LOADED_CHECKPOINT["model_state_dict"])

    print("Modelo c/ checkpoint carregado com sucesso.")

Modelo c/ checkpoint carregado com sucesso.


# Carregando Imagens

In [8]:
from torchvision import transforms
import numpy as np

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [.485, .456, .406], 
        [.229, .224, .225]),
])

torch.manual_seed(42)
np.random.seed(42)

In [10]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

DATA_DIR = "/media/kaua-matheus/HD3201/Data/Agroscope/Teste_Especialista/Corn"
BATCH_SIZE = 32

full_ds = ImageFolder(DATA_DIR, transform=eval_tf)
num_classes = len(full_ds.classes)

In [11]:
test_loader  = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Testes

In [13]:
from sklearn.metrics import f1_score, balanced_accuracy_score
import torch.nn as nn

In [14]:
criterion = nn.CrossEntropyLoss()

In [15]:
NET.to(device)
NET.eval()
total_loss, y_true, y_pred = 0.0, [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = NET(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

avg_loss = total_loss / len(test_loader.dataset)
acc = (np.array(y_true) == np.array(y_pred)).mean()
f1m = f1_score(y_true, y_pred, average="macro")
bacc = balanced_accuracy_score(y_true, y_pred)
    
print(f"""avg_loss: {avg_loss}
acc: {acc}
f1m: {f1m}
bacc: {bacc}
""")

avg_loss: 0.9215367436408997
acc: 0.7142857142857143
f1m: 0.696886446886447
bacc: 0.7166666666666667

